# Natural Language Processing with Disaster Tweets

**Goal:** Predict whether a tweet is about a real disaster (1) or not (0).

**Metric:** F1 score

**Kaggle link:** https://www.kaggle.com/competitions/nlp-getting-started

## 1. Setup & Imports

In [ ]:
# Standard library
import sys
import warnings
from pathlib import Path

# Add project root to Python path so we can import from src/ (ADR-025)
sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

from src.features import build_feature_matrix
from src.text import add_text_features

# Settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

%matplotlib inline

## 2. Configuration

In [ ]:
# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../outputs/models")
SUBMISSIONS_DIR = Path("../outputs/submissions")

# Competition settings
COMPETITION_NAME = "nlp-getting-started"
TARGET_COL = "target"
ID_COL = "id"
TEXT_COL = "text"
RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_TFIDF_FEATURES = 5000
MAX_CHAR_TFIDF_FEATURES = 3000

## 3. Load Data

In [ ]:
train_df = pd.read_csv(DATA_RAW / "train.csv")
test_df = pd.read_csv(DATA_RAW / "test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns:  {list(test_df.columns)}")
print(f"\nTarget distribution:\n{train_df[TARGET_COL].value_counts()}")
print(f"Target balance (% positive): {train_df[TARGET_COL].mean():.3f}")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Overview
train_df.head()

In [ ]:
# Data types and non-null counts
train_df.info()

In [ ]:
# Statistical summary
train_df.describe()

In [ ]:
# Missing values
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({"count": missing, "percentage": missing_pct})
print(missing_df[missing_df["count"] > 0].sort_values("percentage", ascending=False))

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = train_df[TARGET_COL].value_counts()
counts.plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title(f"Target Distribution: {TARGET_COL}")
ax.set_ylabel("Count")
ax.set_xticklabels(["Not Disaster (0)", "Disaster (1)"], rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 50, f"{v} ({v / len(train_df) * 100:.1f}%)", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Text length and word count distributions by target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df["_text_len"] = train_df[TEXT_COL].str.len()
train_df["_word_count"] = train_df[TEXT_COL].str.split().str.len()

for label in [0, 1]:
    subset = train_df[train_df[TARGET_COL] == label]
    axes[0].hist(subset["_text_len"], bins=50, alpha=0.5, label=f"Target={label}")
axes[0].set_title("Tweet Character Length by Target")
axes[0].set_xlabel("Characters")
axes[0].legend()

for label in [0, 1]:
    subset = train_df[train_df[TARGET_COL] == label]
    axes[1].hist(subset["_word_count"], bins=30, alpha=0.5, label=f"Target={label}")
axes[1].set_title("Tweet Word Count by Target")
axes[1].set_xlabel("Words")
axes[1].legend()

plt.tight_layout()
plt.show()

# Drop temporary EDA columns
train_df.drop(columns=["_text_len", "_word_count"], inplace=True)

In [ ]:
# Keyword analysis
keyword_stats = train_df.groupby("keyword")[TARGET_COL].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(f"Unique keywords: {train_df['keyword'].nunique()}")
print(f"Missing keywords: {train_df['keyword'].isna().sum()} ({train_df['keyword'].isna().mean() * 100:.1f}%)")
print(f"Missing locations: {train_df['location'].isna().sum()} ({train_df['location'].isna().mean() * 100:.1f}%)")
print(f"\nTop 10 disaster keywords:\n{keyword_stats.head(10)}")
print(f"\nTop 10 non-disaster keywords:\n{keyword_stats.tail(10)}")

### EDA Key Findings

1. **Target:** Slightly imbalanced — 57% not disaster, 43% disaster
2. **Text:** Disaster tweets tend to be slightly longer (both character and word count)
3. **Keywords:** 221 unique values, only 0.8% missing. Strong signal — some keywords (derailment, debris, wreckage) are 100% disaster, others (aftershock, body bags, blazing) are near 0%
4. **Location:** 33% missing, 3341 unique values — noisy and high-cardinality, likely low value for a first baseline
5. **Missing values:** Only `keyword` (61) and `location` (2533) have missing values. `text` and `target` are complete
6. **Implication for cleaning:** URL-decode keywords (%20 encoding), handle missing keywords, clean text (URLs, special chars). Drop `location` for baseline

## 5. Data Cleaning

In [ ]:
# Apply text cleaning and add text features to both train and test
# WHY: Same transformations on both sets to avoid train/test mismatch (ADR-019)
train_df = add_text_features(train_df, text_col=TEXT_COL)
test_df = add_text_features(test_df, text_col=TEXT_COL)

print("Sample cleaned text:")
print(train_df[["text", "text_clean"]].head(3).to_string())
print("\nKeyword decoding sample:")
print(train_df[train_df["keyword"].notna()][["keyword", "keyword_clean"]].drop_duplicates().head(5).to_string())
new_cols = ["text_clean", "text_len", "word_count", "keyword_clean", "mention_count", "hashtag_count"]
print(f"\nNew columns added: {new_cols}")
print("\nMention/hashtag stats (train):")
print(f"  mention_count: mean={train_df['mention_count'].mean():.2f}, max={train_df['mention_count'].max()}")
print(f"  hashtag_count: mean={train_df['hashtag_count'].mean():.2f}, max={train_df['hashtag_count'].max()}")

In [ ]:
# Verification: no unexpected missing values after cleaning
assert train_df["text_clean"].isna().sum() == 0, "text_clean has NaN"
assert test_df["text_clean"].isna().sum() == 0, "test text_clean has NaN"
assert train_df["keyword_clean"].isna().sum() == 0, "keyword_clean has NaN"
assert test_df["keyword_clean"].isna().sum() == 0, "test keyword_clean has NaN"

print(f"Train text_clean: {train_df['text_clean'].isna().sum()} missing")
print(f"Test text_clean:  {test_df['text_clean'].isna().sum()} missing")
print(f"Train keyword_clean: {train_df['keyword_clean'].isna().sum()} missing")
print(f"Test keyword_clean:  {test_df['keyword_clean'].isna().sum()} missing")
print("\nAll clean — no missing values in processed columns.")

## 6. Feature Engineering

In [ ]:
# Build feature matrices — fit on train only, transform both (ADR-019)
X_train_full, X_test_feat, tfidf, char_tfidf, kw_enc, num_feat = build_feature_matrix(
    train_df, test_df, max_features=MAX_TFIDF_FEATURES, char_max_features=MAX_CHAR_TFIDF_FEATURES
)
y = train_df[TARGET_COL]

print(f"Train feature matrix: {X_train_full.shape}")
print(f"Test feature matrix:  {X_test_feat.shape}")
print(f"Word TF-IDF features: {len(tfidf.vectorizer_.vocabulary_)}")
print(f"Char TF-IDF features: {len(char_tfidf.vectorizer_.vocabulary_)}")
print(f"Keyword features: {len(kw_enc.vectorizer_.vocabulary_)}")
print(f"Numeric features: {len(num_feat.cols)} {num_feat.cols}")
print(f"\nTotal features: {X_train_full.shape[1]}")

In [ ]:
# Verification: feature matrices are consistent
assert X_train_full.shape[1] == X_test_feat.shape[1], "Train/test feature count mismatch"
assert X_train_full.shape[0] == len(train_df), "Train row count mismatch"
assert X_test_feat.shape[0] == len(test_df), "Test row count mismatch"
assert not np.isnan(X_train_full.toarray()).any(), "NaN in train features"

print(f"Train: {X_train_full.shape[0]} rows x {X_train_full.shape[1]} features")
print(f"Test:  {X_test_feat.shape[0]} rows x {X_test_feat.shape[1]} features")
print("Feature matrices verified — no NaN, consistent dimensions.")

## 7. Modeling

In [ ]:
# WHY LogisticRegression: recommended baseline for NLP text classification (WORKFLOW model guide)
# WHY default C=1.0: baseline first, no tuning (ADR-020 YAGNI)
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)

# Cross-validation with StratifiedKFold (ADR-021)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train_full, y, cv=cv, scoring="f1")

print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

### 7.1 Hyperparameter Tuning — GridSearchCV

Search over C, solver, and class_weight to find the best LogisticRegression configuration.

In [ ]:
# WHY GridSearchCV: exhaustive search over a small grid is tractable here (~20 combos)
# WHY these params: C controls regularization strength, solver affects convergence,
# class_weight handles the 57/43 imbalance
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"],
    "class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_full, y)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV F1:  {grid_search.best_score_:.4f}")
print(f"Baseline CV F1: {cv_scores.mean():.4f}")
print(f"Improvement:    {grid_search.best_score_ - cv_scores.mean():+.4f}")

# Use best estimator going forward
model = grid_search.best_estimator_

### 7.2 Model Comparison — XGBoost & LightGBM

Compare gradient boosting models against the LogisticRegression baseline using default hyperparameters (baseline first, ADR-020).

In [ ]:
# WHY default params: baseline first (ADR-020), tune only if the model family shows promise
# WHY both XGBoost and LightGBM: different implementations, one may suit sparse TF-IDF better
# WHY n_estimators=100: keep CV tractable on ~8k sparse features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "LogReg (baseline)": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0),
    "LogReg (tuned)": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **grid_search.best_params_),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, verbose=-1),
}

results = {}
for name, mdl in models.items():
    scores = cross_val_score(mdl, X_train_full, y, cv=cv, scoring="f1")
    results[name] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}
    print(f"{name:20s}  CV F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Pick best model
best_name = max(results, key=lambda k: results[k]["mean"])
print(f"\nBest model: {best_name} (CV F1: {results[best_name]['mean']:.4f})")
model = models[best_name]

### 7.3 Feature Selection — SelectKBest

Test whether dropping low-signal features improves the best model. Use chi2 scoring (compatible with non-negative TF-IDF features) at multiple thresholds.

In [ ]:
# WHY chi2: works with non-negative features (TF-IDF values are >= 0)
# WHY multiple k values: find the sweet spot between noise reduction and signal loss
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
best_model_params = grid_search.best_params_

k_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, X_train_full.shape[1]]
selection_results = {}

for k in k_values:
    if k >= X_train_full.shape[1]:
        X_selected = X_train_full
        label = f"all ({X_train_full.shape[1]})"
    else:
        selector = SelectKBest(chi2, k=k)
        X_selected = selector.fit_transform(X_train_full, y)
        label = str(k)
    mdl = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **best_model_params)
    scores = cross_val_score(mdl, X_selected, y, cv=cv, scoring="f1")
    selection_results[label] = {"mean": scores.mean(), "std": scores.std()}
    print(f"k={label:>10s}  CV F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Find best k
best_k = max(selection_results, key=lambda k: selection_results[k]["mean"])
print(f"\nBest k: {best_k} (CV F1: {selection_results[best_k]['mean']:.4f})")
print(f"Full features CV F1: {selection_results[f'all ({X_train_full.shape[1]})']['mean']:.4f}")
diff = selection_results[best_k]["mean"] - selection_results[f"all ({X_train_full.shape[1]})"]["mean"]
print(f"Difference: {diff:+.4f}")

# Apply best selection if it improves over full features
if best_k != f"all ({X_train_full.shape[1]})":
    best_k_int = int(best_k)
    selector_final = SelectKBest(chi2, k=best_k_int)
    X_train_full = selector_final.fit_transform(X_train_full, y)
    X_test_feat = selector_final.transform(X_test_feat)
    print(f"\nApplied SelectKBest(k={best_k_int}): {X_train_full.shape[1]} features retained")
else:
    selector_final = None
    print("\nNo feature selection applied — full feature set is optimal")

In [ ]:
# Train best model on a hold-out split for sanity-check evaluation
# WHY hold-out after GridSearchCV on full data: this is NOT an independent evaluation —
# the val set was seen during CV. It serves as a sanity check only.
# The primary metric is the CV F1 from GridSearchCV (ADR-021).
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
model.fit(X_train, y_train)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}")
print(f"Train target balance: {y_train.mean():.3f}")
print(f"Val target balance:   {y_val.mean():.3f}")

## 8. Evaluation

In [ ]:
# Validation set evaluation
y_pred = model.predict(X_val)

print(f"Validation F1: {f1_score(y_val, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Not Disaster", "Disaster"]))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=["Not Disaster", "Disaster"],
    yticklabels=["Not Disaster", "Disaster"],
)
ax.set_title("Confusion Matrix")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# Top features analysis (model-dependent)
feature_names = tfidf.vectorizer_.get_feature_names_out().tolist()
feature_names += [f"char:{n}" for n in char_tfidf.vectorizer_.get_feature_names_out()]
feature_names += kw_enc.vectorizer_.get_feature_names_out().tolist()
feature_names += num_feat.cols

if hasattr(model, "coef_"):
    coefs = model.coef_[0]
    top_positive = sorted(zip(feature_names, coefs, strict=False), key=lambda x: x[1], reverse=True)[:15]
    top_negative = sorted(zip(feature_names, coefs, strict=False), key=lambda x: x[1])[:15]
    print("Top 15 features predicting DISASTER:")
    for name, coef in top_positive:
        print(f"  {name:30s} {coef:+.4f}")
    print("\nTop 15 features predicting NOT DISASTER:")
    for name, coef in top_negative:
        print(f"  {name:30s} {coef:+.4f}")
elif hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    top_features = sorted(zip(feature_names, importances, strict=False), key=lambda x: x[1], reverse=True)[:20]
    print("Top 20 features by importance:")
    for name, imp in top_features:
        print(f"  {name:30s} {imp:.4f}")
else:
    print("No feature importance available for this model type.")

## 9. Submission

In [ ]:
# Retrain best model from comparison on 100% of training data
# WHY: Use all available labeled data to maximize signal for the test predictions
model_final = models[best_name]
model_final.fit(X_train_full, y)
print(f"Final model: {best_name}")
print(f"Trained on {X_train_full.shape[0]} samples, {X_train_full.shape[1]} features")
if selector_final is not None:
    print(f"Feature selection: SelectKBest(k={best_k})")

# Generate predictions on test set
test_predictions = model_final.predict(X_test_feat)
print(f"Test predictions: {len(test_predictions)} samples")
print(f"Predicted distribution: {pd.Series(test_predictions).value_counts().to_dict()}")

# Create submission DataFrame matching sample_submission.csv format (id, target)
submission = pd.DataFrame(
    {
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_predictions,
    }
)

# Validate format against sample submission
sample_sub = pd.read_csv(DATA_RAW / "sample_submission.csv")
assert list(submission.columns) == list(sample_sub.columns), "Column mismatch with sample submission"
assert len(submission) == len(sample_sub), f"Row count mismatch: {len(submission)} vs {len(sample_sub)}"
assert set(submission[TARGET_COL].unique()).issubset({0, 1}), "Unexpected target values"
print("Format validated - matches sample_submission.csv")

# Only save a new CSV if predictions differ from the latest submission
latest_sub = pd.read_csv(SUBMISSIONS_DIR / "submission_003.csv")
if submission[TARGET_COL].equals(latest_sub[TARGET_COL]):
    print("Predictions unchanged vs submission_003.csv — no new file generated.")
else:
    submission_path = SUBMISSIONS_DIR / "submission_004.csv"
    submission.to_csv(submission_path, index=False)
    print(f"New submission saved: {submission_path}")
    print(f"Shape: {submission.shape}")

submission.head(10)